In [ ]:
# =========================================
# PHASE 3: FEATURE ENGINEERING & PREPROCESSING
# CLEAN NUMPY-ONLY VERSION
# NEW NOTEBOOK / NEW OUTPUT FOLDER
# =========================================

# =========================================================
# STEP 0: MOUNT DRIVE
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# STEP 1: IMPORTS
# =========================================================
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import pickle
import gc
import re

# =========================================================
# STEP 2: CONFIGURATION
# =========================================================
MAX_LEN = 50
CHUNK_SIZE = 100   # keep small for Colab safety

DATA_PATH = Path("/content/drive/MyDrive/Instacart")
PHASE2_PATH = DATA_PATH / "phase2_outputs"

# IMPORTANT: use a NEW clean folder
PHASE3_PATH = DATA_PATH / "phase3_outputs_final"
PHASE3_PATH.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print("Data folder      :", DATA_PATH)
print("Phase 2 folder   :", PHASE2_PATH)
print("Phase 3 folder   :", PHASE3_PATH)
print("RUN_ID           :", RUN_ID)
print("MAX_LEN          :", MAX_LEN)
print("CHUNK_SIZE       :", CHUNK_SIZE)

# =========================================================
# STEP 3: HELPER FUNCTIONS
# =========================================================
def save_pickle(obj, filename, folder=PHASE3_PATH):
    """
    Save object to pickle file in the chosen folder.
    """
    path = folder / filename
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved -> {path}")
    return path

def load_pickle(filename, folder=PHASE3_PATH):
    """
    Load exact filename if it exists.
    """
    path = folder / filename
    if not path.exists():
        return None
    with open(path, "rb") as f:
        print(f"Loaded exact -> {path}")
        return pickle.load(f)

def save_timestamped(obj, base_name, folder=PHASE3_PATH):
    """
    Save with timestamped filename:
    <base_name>_YYYYMMDD_HHMMSS.pkl
    """
    filename = f"{base_name}_{RUN_ID}.pkl"
    return save_pickle(obj, filename, folder=folder)

def get_latest_timestamped_file(base_name, folder=PHASE3_PATH):
    """
    Find latest timestamped file matching:
    <base_name>_YYYYMMDD_HHMMSS.pkl
    """
    files = list(folder.glob(f"{base_name}_*.pkl"))
    if not files:
        return None

    valid_files = []
    pattern = re.compile(rf"^{re.escape(base_name)}_(\d{{8}}_\d{{6}})\.pkl$")

    for f in files:
        m = pattern.match(f.name)
        if m:
            valid_files.append((m.group(1), f))

    if not valid_files:
        return None

    valid_files.sort(key=lambda x: x[0], reverse=True)
    return valid_files[0][1]

def load_phase_file(base_name, folder, allow_stable_fallback=True):
    """
    1. Try latest timestamped file
    2. Fallback to stable file <base_name>.pkl
    """
    latest = get_latest_timestamped_file(base_name, folder=folder)
    if latest is not None:
        with open(latest, "rb") as f:
            print(f"Loaded latest timestamped -> {latest}")
            return pickle.load(f)

    if allow_stable_fallback:
        stable = folder / f"{base_name}.pkl"
        if stable.exists():
            with open(stable, "rb") as f:
                print(f"Loaded stable fallback -> {stable}")
                return pickle.load(f)

    print(f"No saved file found for: {base_name}")
    return None

def checkpoint_validate(condition, message):
    """
    Small helper for readable assertions.
    """
    if not condition:
        raise AssertionError(message)

def pad_sequences_simple(sequences, maxlen, value=0, dtype="int32"):
    """
    NumPy-only replacement for keras pad_sequences.
    - left truncation
    - left padding
    """
    arr = np.full((len(sequences), maxlen), value, dtype=dtype)
    for i, seq in enumerate(sequences):
        seq = seq[-maxlen:]  # keep only last maxlen values
        if len(seq) > 0:
            arr[i, -len(seq):] = seq
    return arr

# =========================================================
# STEP 4: LOAD PHASE 2 FILES
# =========================================================
print("\n================ STEP 4: LOAD PHASE 2 FILES ================")

train_data = load_phase_file("train_data", PHASE2_PATH)
test_data = load_phase_file("test_data", PHASE2_PATH)
product2id = load_phase_file("product2id", PHASE2_PATH)

if train_data is None or test_data is None or product2id is None:
    raise ValueError("Missing required Phase 2 files: train_data / test_data / product2id")

train_users = set(train_data.keys())
test_users = set(test_data.keys())
all_phase2_users = train_users.union(test_users)

print("Train users:", len(train_users))
print("Test users :", len(test_users))
print("Total users:", len(all_phase2_users))
print("Product vocab size:", len(product2id))

# ---------------- VALIDATION: STEP 4 ----------------
print("\nSTEP 4 VALIDATION")
checkpoint_validate(len(train_users) > 0, "Train users empty")
checkpoint_validate(len(test_users) > 0, "Test users empty")
checkpoint_validate(len(train_users.intersection(test_users)) == 0, "Train/test overlap found")
checkpoint_validate("<UNK>" in product2id, "<UNK> missing in product2id")
checkpoint_validate(product2id["<UNK>"] == 0, "<UNK> token must map to 0")
print("Step 4 validation passed.")

# =========================================================
# STEP 5: LOAD RAW FILES
# =========================================================
print("\n================ STEP 5: LOAD RAW FILES ================")

orders = pd.read_csv(
    DATA_PATH / "orders.csv",
    usecols=["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"],
    dtype={
        "order_id": "int32",
        "user_id": "int32",
        "order_number": "int16",
        "order_dow": "int8",
        "order_hour_of_day": "int8",
        "days_since_prior_order": "float32"
    }
)

prior = pd.read_csv(
    DATA_PATH / "order_products__prior.csv",
    usecols=["order_id", "product_id", "add_to_cart_order"],
    dtype={
        "order_id": "int32",
        "product_id": "int32",
        "add_to_cart_order": "int16"
    }
)

products = pd.read_csv(
    DATA_PATH / "products.csv",
    usecols=["product_id", "aisle_id", "department_id"],
    dtype={
        "product_id": "int32",
        "aisle_id": "int16",
        "department_id": "int16"
    }
)

print("orders shape  :", orders.shape)
print("prior shape   :", prior.shape)
print("products shape:", products.shape)

# ---------------- VALIDATION: STEP 5 ----------------
print("\nSTEP 5 VALIDATION")
print("orders nulls:\n", orders.isnull().sum())
print("prior nulls:\n", prior.isnull().sum())
print("products nulls:\n", products.isnull().sum())

checkpoint_validate("order_dow" in orders.columns, "order_dow missing")
checkpoint_validate("order_hour_of_day" in orders.columns, "order_hour_of_day missing")
checkpoint_validate("days_since_prior_order" in orders.columns, "days_since_prior_order missing")
checkpoint_validate("aisle_id" in products.columns, "aisle_id missing")
checkpoint_validate("department_id" in products.columns, "department_id missing")
print("Step 5 validation passed.")

# =========================================================
# STEP 6: TEMPORAL HOUSEKEEPING
# =========================================================
print("\n================ STEP 6: TEMPORAL HOUSEKEEPING ================")

orders["days_since_prior_order"] = orders["days_since_prior_order"].fillna(0)
orders["is_first_order"] = (orders["order_number"] == 1).astype("int8")

# ---------------- VALIDATION: STEP 6 ----------------
print("\nSTEP 6 VALIDATION")
checkpoint_validate(orders["days_since_prior_order"].isnull().sum() == 0, "days_since_prior_order still has nulls")
checkpoint_validate(set(orders["is_first_order"].unique()).issubset({0, 1}), "is_first_order has invalid values")
print("Missing days_since_prior_order:", orders["days_since_prior_order"].isnull().sum())
print("is_first_order values:", orders["is_first_order"].value_counts().to_dict())
print("Step 6 validation passed.")

# =========================================================
# STEP 7: BUILD MASTER TABLE
# =========================================================
print("\n================ STEP 7: BUILD MASTER TABLE ================")

master_df = load_phase_file("master_df", PHASE3_PATH)

if master_df is None:
    master_df = (
        prior
        .merge(orders, on="order_id", how="left")
        .merge(products, on="product_id", how="left")
    )

    # Keep only Phase 2 users
    master_df = master_df[master_df["user_id"].isin(all_phase2_users)]

    # Chronological order
    master_df = master_df.sort_values(
        ["user_id", "order_number", "add_to_cart_order"]
    ).reset_index(drop=True)

    save_timestamped(master_df, "master_df")
    save_pickle(master_df, "master_df.pkl")

print("master_df shape:", master_df.shape)
display(master_df.head())

del prior, orders, products
gc.collect()

# ---------------- VALIDATION: STEP 7 ----------------
print("\nSTEP 7 VALIDATION")
required_cols = [
    "order_id", "product_id", "add_to_cart_order", "user_id", "order_number",
    "order_dow", "order_hour_of_day", "days_since_prior_order",
    "is_first_order", "aisle_id", "department_id"
]

for c in required_cols:
    checkpoint_validate(c in master_df.columns, f"Missing column: {c}")

print(master_df[required_cols].isnull().sum())

checkpoint_validate(master_df["user_id"].isnull().sum() == 0, "user_id has nulls")
checkpoint_validate(master_df["product_id"].isnull().sum() == 0, "product_id has nulls")
checkpoint_validate(set(master_df["user_id"].unique()).issubset(all_phase2_users), "master_df has unknown users")
print("Step 7 validation passed.")

# =========================================================
# STEP 8: BUILD CATEGORY ENCODERS
# =========================================================
print("\n================ STEP 8: BUILD CATEGORY ENCODERS ================")

aisle2id = load_phase_file("aisle2id", PHASE3_PATH)
dept2id = load_phase_file("dept2id", PHASE3_PATH)

if aisle2id is None or dept2id is None:
    unique_aisles = sorted(master_df["aisle_id"].dropna().unique())
    unique_depts = sorted(master_df["department_id"].dropna().unique())

    aisle2id = {a: i + 1 for i, a in enumerate(unique_aisles)}
    dept2id = {d: i + 1 for i, d in enumerate(unique_depts)}

    save_timestamped(aisle2id, "aisle2id")
    save_timestamped(dept2id, "dept2id")
    save_pickle(aisle2id, "aisle2id.pkl")
    save_pickle(dept2id, "dept2id.pkl")

print("Aisle vocab size     :", len(aisle2id))
print("Department vocab size:", len(dept2id))

# ---------------- VALIDATION: STEP 8 ----------------
print("\nSTEP 8 VALIDATION")
checkpoint_validate(len(aisle2id) > 0, "aisle2id empty")
checkpoint_validate(len(dept2id) > 0, "dept2id empty")
print("Sample aisle mappings:", list(aisle2id.items())[:5])
print("Sample dept mappings :", list(dept2id.items())[:5])
print("Step 8 validation passed.")

# =========================================================
# STEP 9: ENCODE MASTER TABLE
# =========================================================
print("\n================ STEP 9: ENCODE MASTER TABLE ================")

encoded_master_df = load_phase_file("encoded_master_df", PHASE3_PATH)

if encoded_master_df is None:
    encoded_master_df = master_df.copy()

    encoded_master_df["product_enc"] = encoded_master_df["product_id"].map(
        lambda x: product2id.get(x, 0)
    ).astype("int32")

    encoded_master_df["aisle_enc"] = encoded_master_df["aisle_id"].map(
        lambda x: aisle2id.get(x, 0)
    ).astype("int16")

    encoded_master_df["dept_enc"] = encoded_master_df["department_id"].map(
        lambda x: dept2id.get(x, 0)
    ).astype("int16")

    encoded_master_df["dow_enc"] = encoded_master_df["order_dow"].astype("int8")
    encoded_master_df["hour_enc"] = encoded_master_df["order_hour_of_day"].astype("int8")
    encoded_master_df["days_enc"] = encoded_master_df["days_since_prior_order"].astype("float32")
    encoded_master_df["first_order_enc"] = encoded_master_df["is_first_order"].astype("int8")

    save_timestamped(encoded_master_df, "encoded_master_df")
    save_pickle(encoded_master_df, "encoded_master_df.pkl")

print("encoded_master_df shape:", encoded_master_df.shape)

# ---------------- VALIDATION: STEP 9 ----------------
print("\nSTEP 9 VALIDATION")
enc_cols = ["product_enc", "aisle_enc", "dept_enc", "dow_enc", "hour_enc", "days_enc", "first_order_enc"]
print(encoded_master_df[enc_cols].isnull().sum())

checkpoint_validate(encoded_master_df["product_enc"].isnull().sum() == 0, "product_enc has nulls")
checkpoint_validate(encoded_master_df["aisle_enc"].isnull().sum() == 0, "aisle_enc has nulls")
checkpoint_validate(encoded_master_df["dept_enc"].isnull().sum() == 0, "dept_enc has nulls")
checkpoint_validate(encoded_master_df["product_enc"].min() >= 0, "product_enc negative")
checkpoint_validate(encoded_master_df["aisle_enc"].min() >= 0, "aisle_enc negative")
checkpoint_validate(encoded_master_df["dept_enc"].min() >= 0, "dept_enc negative")
print("Step 9 validation passed.")

# =========================================================
# STEP 10: BUILD USER EVENT SEQUENCES
# =========================================================
print("\n================ STEP 10: BUILD USER EVENT SEQUENCES ================")

user_sequences = load_phase_file("user_sequences_phase3", PHASE3_PATH)

if user_sequences is None:
    user_sequences = {}

    for user_id, grp in encoded_master_df.groupby("user_id", sort=False):
        grp = grp.sort_values(["order_number", "add_to_cart_order"])

        user_sequences[user_id] = {
            "p": grp["product_enc"].tolist(),
            "a": grp["aisle_enc"].tolist(),
            "d": grp["dept_enc"].tolist(),
            "dow": grp["dow_enc"].tolist(),
            "hr": grp["hour_enc"].tolist(),
            "days": grp["days_enc"].tolist()
        }

    save_timestamped(user_sequences, "user_sequences_phase3")
    save_pickle(user_sequences, "user_sequences_phase3.pkl")

print("Users in user_sequences:", len(user_sequences))

# ---------------- VALIDATION: STEP 10 ----------------
print("\nSTEP 10 VALIDATION")
checkpoint_validate(len(user_sequences) == len(all_phase2_users), "User count mismatch in Phase 3 sequences")

sample_user = list(user_sequences.keys())[0]
sample_seq = user_sequences[sample_user]

print("Sample user:", sample_user)
for k, v in sample_seq.items():
    print(k, len(v), v[:5])

sample_lengths = [len(v) for v in sample_seq.values()]
checkpoint_validate(len(set(sample_lengths)) == 1, "Sample sequence features misaligned")

for u in list(user_sequences.keys())[:50]:
    lengths = [len(v) for v in user_sequences[u].values()]
    checkpoint_validate(len(set(lengths)) == 1, f"Misaligned sequence lengths for user {u}")

print("Step 10 validation passed.")

# =========================================================
# STEP 11: BUILD BATCHES SAFELY (WITH RESUME)
# =========================================================
print("\n================ STEP 11: BUILD BATCHES SAFELY (WITH RESUME) ================")

def build_padded_batch_for_users(user_ids_batch, user_sequences, max_len=50):
    """
    Convert a small list of users into one padded supervised batch.
    """
    Xp, Xa, Xd, Xdow, Xhr, Xdays, y = [], [], [], [], [], [], []

    for u in user_ids_batch:
        if u not in user_sequences:
            continue

        seq = user_sequences[u]["p"]
        if len(seq) < 2:
            continue

        for t in range(1, len(seq)):
            target = seq[t]

            # Skip unknown target
            if target == 0:
                continue

            start = max(0, t - max_len)

            Xp.append(user_sequences[u]["p"][start:t])
            Xa.append(user_sequences[u]["a"][start:t])
            Xd.append(user_sequences[u]["d"][start:t])
            Xdow.append(user_sequences[u]["dow"][start:t])
            Xhr.append(user_sequences[u]["hr"][start:t])
            Xdays.append(user_sequences[u]["days"][start:t])
            y.append(target)

    if len(y) == 0:
        return None

    batch = {
        "Xp": pad_sequences_simple(Xp, maxlen=max_len, value=0, dtype="int32"),
        "Xa": pad_sequences_simple(Xa, maxlen=max_len, value=0, dtype="int16"),
        "Xd": pad_sequences_simple(Xd, maxlen=max_len, value=0, dtype="int16"),
        "Xdow": pad_sequences_simple(Xdow, maxlen=max_len, value=0, dtype="int8"),
        "Xhr": pad_sequences_simple(Xhr, maxlen=max_len, value=0, dtype="int8"),
        "Xdays": pad_sequences_simple(Xdays, maxlen=max_len, value=0.0, dtype="float32"),
        "y": np.array(y, dtype="int32")
    }
    return batch

def get_completed_batch_numbers(dataset_name, folder=PHASE3_PATH):
    """
    Detect already-saved batch numbers for resume.
    """
    completed = set()

    for f in folder.glob(f"{dataset_name}_batch_*.pkl"):
        m = re.match(rf"{dataset_name}_batch_(\d+)_\d{{8}}_\d{{6}}\.pkl$", f.name)
        if m:
            completed.add(int(m.group(1)))

    return completed

def process_users_in_small_chunks(user_ids, user_sequences, dataset_name, max_len=50, chunk_size=100):
    """
    Process users in small chunks and save one batch file at a time.
    """
    batch_files = []
    user_ids = list(user_ids)

    completed_batches = get_completed_batch_numbers(dataset_name, PHASE3_PATH)
    print(f"Already completed {dataset_name} batches:", len(completed_batches))

    for i in range(0, len(user_ids), chunk_size):
        batch_users = user_ids[i:i + chunk_size]
        batch_num = i // chunk_size + 1

        if batch_num in completed_batches:
            existing_matches = sorted(PHASE3_PATH.glob(f"{dataset_name}_batch_{batch_num:04d}_*.pkl"))
            if existing_matches:
                existing_name = existing_matches[-1].name
                print(f"Skipping {dataset_name} batch {batch_num} (already exists: {existing_name})")
                batch_files.append(existing_name)
            continue

        print(f"Processing {dataset_name} batch {batch_num} | users {i} to {i + len(batch_users) - 1}")

        batch = build_padded_batch_for_users(batch_users, user_sequences, max_len=max_len)

        if batch is not None:
            batch_filename = f"{dataset_name}_batch_{batch_num:04d}_{RUN_ID}.pkl"
            save_pickle(batch, batch_filename, folder=PHASE3_PATH)
            batch_files.append(batch_filename)

        del batch
        gc.collect()

    return batch_files

print("\nCreating / refreshing TRAIN batch index...")
train_batch_files = process_users_in_small_chunks(
    sorted(train_users), user_sequences, "train", MAX_LEN, CHUNK_SIZE
)

train_batch_index = {"batch_files": train_batch_files}
save_timestamped(train_batch_index, "train_batch_index")
save_pickle(train_batch_index, "train_batch_index.pkl")

print("\nCreating / refreshing TEST batch index...")
test_batch_files = process_users_in_small_chunks(
    sorted(test_users), user_sequences, "test", MAX_LEN, CHUNK_SIZE
)

test_batch_index = {"batch_files": test_batch_files}
save_timestamped(test_batch_index, "test_batch_index")
save_pickle(test_batch_index, "test_batch_index.pkl")

# ---------------- VALIDATION: STEP 11 ----------------
print("\nSTEP 11 VALIDATION")
print("Train batch files:", len(train_batch_index["batch_files"]))
print("Test batch files :", len(test_batch_index["batch_files"]))

checkpoint_validate(len(train_batch_index["batch_files"]) > 0, "No train batches created")
checkpoint_validate(len(test_batch_index["batch_files"]) > 0, "No test batches created")
print("Step 11 validation passed.")

# =========================================================
# STEP 12: VALIDATE ONE SAMPLE BATCH
# =========================================================
print("\n================ STEP 12: VALIDATE SAMPLE BATCH ================")

sample_train_batch_file = train_batch_index["batch_files"][0]
sample_batch = load_pickle(sample_train_batch_file, PHASE3_PATH)

required_batch_keys = ["Xp", "Xa", "Xd", "Xdow", "Xhr", "Xdays", "y"]
for k in required_batch_keys:
    checkpoint_validate(k in sample_batch, f"Missing batch key: {k}")

print("Sample batch file:", sample_train_batch_file)
for k, v in sample_batch.items():
    print(k, v.shape)

n = sample_batch["y"].shape[0]
checkpoint_validate(sample_batch["Xp"].shape[0] == n, "Xp row mismatch")
checkpoint_validate(sample_batch["Xa"].shape[0] == n, "Xa row mismatch")
checkpoint_validate(sample_batch["Xd"].shape[0] == n, "Xd row mismatch")
checkpoint_validate(sample_batch["Xdow"].shape[0] == n, "Xdow row mismatch")
checkpoint_validate(sample_batch["Xhr"].shape[0] == n, "Xhr row mismatch")
checkpoint_validate(sample_batch["Xdays"].shape[0] == n, "Xdays row mismatch")

checkpoint_validate(sample_batch["Xp"].shape[1] == MAX_LEN, "Xp MAX_LEN mismatch")
checkpoint_validate(sample_batch["Xa"].shape[1] == MAX_LEN, "Xa MAX_LEN mismatch")
checkpoint_validate(sample_batch["Xd"].shape[1] == MAX_LEN, "Xd MAX_LEN mismatch")
checkpoint_validate(sample_batch["Xdow"].shape[1] == MAX_LEN, "Xdow MAX_LEN mismatch")
checkpoint_validate(sample_batch["Xhr"].shape[1] == MAX_LEN, "Xhr MAX_LEN mismatch")
checkpoint_validate(sample_batch["Xdays"].shape[1] == MAX_LEN, "Xdays MAX_LEN mismatch")

print("Step 12 validation passed.")

# =========================================================
# STEP 13: TARGET + VALUE VALIDATION
# =========================================================
print("\n================ STEP 13: TARGET + VALUE VALIDATION ================")

y = sample_batch["y"]
print("Sample targets:", y[:10])
print("Min target:", y.min())
print("Max target:", y.max())

checkpoint_validate(np.all(y >= 1), "Targets contain 0 or negative values")
checkpoint_validate(np.max(y) <= max(product2id.values()), "Targets exceed vocab range")

checkpoint_validate(np.min(sample_batch["Xp"]) >= 0, "Xp contains negative values")
checkpoint_validate(np.min(sample_batch["Xa"]) >= 0, "Xa contains negative values")
checkpoint_validate(np.min(sample_batch["Xd"]) >= 0, "Xd contains negative values")
checkpoint_validate(np.min(sample_batch["Xdow"]) >= 0, "Xdow contains negative values")
checkpoint_validate(np.min(sample_batch["Xhr"]) >= 0, "Xhr contains negative values")
checkpoint_validate(np.min(sample_batch["Xdays"]) >= 0, "Xdays contains negative values")

print("Step 13 validation passed.")

# =========================================================
# STEP 14: SAVE PHASE 3 METADATA
# =========================================================
print("\n================ STEP 14: SAVE PHASE 3 METADATA ================")

phase3_metadata = {
    "RUN_ID": RUN_ID,
    "MAX_LEN": MAX_LEN,
    "CHUNK_SIZE": CHUNK_SIZE,
    "num_train_users": len(train_users),
    "num_test_users": len(test_users),
    "num_train_batches": len(train_batch_index["batch_files"]),
    "num_test_batches": len(test_batch_index["batch_files"]),
    "product_vocab_size": len(product2id),
    "aisle_vocab_size": len(aisle2id),
    "dept_vocab_size": len(dept2id),
    "sample_train_batch_file": sample_train_batch_file
}

save_timestamped(phase3_metadata, "phase3_metadata")
save_pickle(phase3_metadata, "phase3_metadata.pkl")

print("Phase 3 metadata saved.")
for k, v in phase3_metadata.items():
    print(f"{k}: {v}")

# ---------------- VALIDATION: STEP 14 ----------------
print("\nSTEP 14 VALIDATION")
checkpoint_validate(phase3_metadata["num_train_users"] > 0, "num_train_users invalid")
checkpoint_validate(phase3_metadata["num_test_users"] > 0, "num_test_users invalid")
checkpoint_validate(phase3_metadata["num_train_batches"] > 0, "num_train_batches invalid")
checkpoint_validate(phase3_metadata["num_test_batches"] > 0, "num_test_batches invalid")
checkpoint_validate(phase3_metadata["product_vocab_size"] > 0, "product_vocab_size invalid")
print("Step 14 validation passed.")

# =========================================================
# FINAL APPROVAL
# =========================================================
print("\n========================================")
print("✅ PHASE 3 FULLY VALIDATED")
print("========================================")
print("Train batches:", len(train_batch_index["batch_files"]))
print("Test batches :", len(test_batch_index["batch_files"]))
print("Sample batch :", sample_train_batch_file)

Mounted at /content/drive
Data folder      : /content/drive/MyDrive/Instacart
Phase 2 folder   : /content/drive/MyDrive/Instacart/phase2_outputs
Phase 3 folder   : /content/drive/MyDrive/Instacart/phase3_outputs_final
RUN_ID           : 20260422_185902
MAX_LEN          : 50
CHUNK_SIZE       : 100

================ STEP 4: LOAD PHASE 2 FILES ================
Loaded latest timestamped -> /content/drive/MyDrive/Instacart/phase2_outputs/train_data_20260415_213453.pkl
Loaded latest timestamped -> /content/drive/MyDrive/Instacart/phase2_outputs/test_data_20260415_213453.pkl
Loaded stable fallback -> /content/drive/MyDrive/Instacart/phase2_outputs/product2id.pkl
Train users: 164331
Test users : 41083
Total users: 205414
Product vocab size: 25001

STEP 4 VALIDATION
Step 4 validation passed.

================ STEP 5: LOAD RAW FILES ================
orders shape  : (3421083, 6)
prior shape   : (32434489, 3)
products shape: (49688, 3)

STEP 5 VALIDATION
orders nulls:
 order_id                    

,order_id,product_id,add_to_cart_order,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order,aisle_id,department_id
0,2539329,196,1,1,1,2,8,0.0,1,77,7
1,2539329,14084,2,1,1,2,8,0.0,1,91,16
2,2539329,12427,3,1,1,2,8,0.0,1,23,19
3,2539329,26088,4,1,1,2,8,0.0,1,23,19
4,2539329,26405,5,1,1,2,8,0.0,1,54,17



STEP 7 VALIDATION
order_id                  0
product_id                0
add_to_cart_order         0
user_id                   0
order_number              0
order_dow                 0
order_hour_of_day         0
days_since_prior_order    0
is_first_order            0
aisle_id                  0
department_id             0
dtype: int64
Step 7 validation passed.

================ STEP 8: BUILD CATEGORY ENCODERS ================
No saved file found for: aisle2id
No saved file found for: dept2id
Saved -> /content/drive/MyDrive/Instacart/phase3_outputs_final/aisle2id_20260422_185902.pkl
Saved -> /content/drive/MyDrive/Instacart/phase3_outputs_final/dept2id_20260422_185902.pkl
Saved -> /content/drive/MyDrive/Instacart/phase3_outputs_final/aisle2id.pkl
Saved -> /content/drive/MyDrive/Instacart/phase3_outputs_final/dept2id.pkl
Aisle vocab size     : 134
Department vocab size: 21

STEP 8 VALIDATION
Sample aisle mappings: [(np.int16(1), 1), (np.int16(2), 2), (np.int16(3), 3), (np.int16(4), 4)